# Field Economics Starter

**Task:** [Replace with task title]
**Date:** [YYYY-MM-DD]

Template for NPV, Monte Carlo sensitivity, and project economics.

In [ ]:
# ── NeqSim Setup (dual-boot: devtools or pip) ──
import importlib, subprocess, sys

try:
    from neqsim_dev_setup import neqsim_init, neqsim_classes
    ns = neqsim_init(recompile=False)
    ns = neqsim_classes(ns)
    NEQSIM_MODE = "devtools"
    print("NeqSim loaded via devtools (local dev mode)")
except Exception:
    try:
        import neqsim
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "neqsim"])
    from neqsim import jneqsim
    NEQSIM_MODE = "pip"
    print("NeqSim loaded via pip package")

import matplotlib.pyplot as plt
import numpy as np
import json, os, pathlib

NOTEBOOK_DIR = pathlib.Path(globals().get(
    "__vsc_ipynb_file__", os.path.abspath("step2_analysis/01_field_economics.ipynb")
)).resolve().parent
TASK_DIR = NOTEBOOK_DIR.parent
FIGURES_DIR = TASK_DIR / "figures"
FIGURES_DIR.mkdir(exist_ok=True)

In [ ]:
# ── Import NeqSim classes ──
import jpype
if NEQSIM_MODE == "pip":
    from neqsim import jneqsim

# Field development classes
try:
    DCFCalculator = jpype.JClass("neqsim.process.util.fielddevelopment.DCFCalculator")
    ProductionProfile = jpype.JClass("neqsim.process.util.fielddevelopment.ProductionProfile")
    print("DCFCalculator and ProductionProfile imported")
except Exception:
    print("Note: DCFCalculator not available; using Python-based NPV")
    DCFCalculator = None
    ProductionProfile = None

## 1. Production Profile

In [ ]:
# ── Define production profile ──
# Example: 15-year gas field
years = np.arange(0, 16)

# Plateau + decline (exponential decline after year 5)
plateau_rate_MSm3_yr = 3.0  # GSm3/yr at plateau
decline_start = 5
decline_rate = 0.15  # 15% per year

production = []
for y in years:
    if y == 0:
        production.append(0.0)  # Development year
    elif y <= decline_start:
        production.append(plateau_rate_MSm3_yr)
    else:
        rate = plateau_rate_MSm3_yr * np.exp(-decline_rate * (y - decline_start))
        production.append(rate)

production = np.array(production)
cumulative = np.cumsum(production)
print("Total production: {:.1f} GSm3 over {} years".format(cumulative[-1], len(years)-1))

## 2. Economic Parameters

In [ ]:
# ── Economic assumptions ──
gas_price = 2.0  # USD/MMBtu (or NOK/Sm3 - adjust units)
conversion_factor = 35.315  # Sm3 to MMBtu (approximate for natural gas)

capex_total = 5000.0  # MUSD total CAPEX
opex_per_unit = 0.5  # USD/MMBtu OPEX
discount_rate = 0.08  # 8% real discount rate

# Revenue and cost calculation
revenue = production * 1e9 / conversion_factor * gas_price / 1e6  # MUSD
opex = production * 1e9 / conversion_factor * opex_per_unit / 1e6  # MUSD

# Simple cash flow (before tax)
cash_flow = np.zeros(len(years))
cash_flow[0] = -capex_total  # Year 0: investment
for i in range(1, len(years)):
    cash_flow[i] = revenue[i] - opex[i]

# Discounted cash flow
discount_factors = 1.0 / (1 + discount_rate) ** years
dcf = cash_flow * discount_factors
npv = np.sum(dcf)

print("NPV (pre-tax): {:.0f} MUSD".format(npv))
print("Total revenue: {:.0f} MUSD".format(np.sum(revenue)))
print("Total OPEX: {:.0f} MUSD".format(np.sum(opex)))

## 3. Monte Carlo Sensitivity Analysis

In [ ]:
# ── Monte Carlo NPV ──
N_SIM = 1000
np.random.seed(42)

# Uncertain parameters (triangular distributions)
gas_prices_mc = np.random.triangular(1.0, 2.0, 4.0, N_SIM)
capex_mult_mc = np.random.triangular(0.85, 1.0, 1.40, N_SIM)
plateau_mc = np.random.triangular(2.0, 3.0, 4.0, N_SIM)

npv_results = []
for i in range(N_SIM):
    # Rebuild production with sampled plateau
    prod_i = []
    for y in years:
        if y == 0:
            prod_i.append(0.0)
        elif y <= decline_start:
            prod_i.append(plateau_mc[i])
        else:
            prod_i.append(plateau_mc[i] * np.exp(-decline_rate * (y - decline_start)))
    prod_i = np.array(prod_i)

    rev_i = prod_i * 1e9 / conversion_factor * gas_prices_mc[i] / 1e6
    opex_i = prod_i * 1e9 / conversion_factor * opex_per_unit / 1e6
    cf_i = np.zeros(len(years))
    cf_i[0] = -capex_total * capex_mult_mc[i]
    cf_i[1:] = rev_i[1:] - opex_i[1:]
    npv_i = np.sum(cf_i * discount_factors)
    npv_results.append(npv_i)

npv_results = np.array(npv_results)

p10 = np.percentile(npv_results, 10)
p50 = np.percentile(npv_results, 50)
p90 = np.percentile(npv_results, 90)
prob_negative = np.mean(npv_results < 0) * 100

print("Monte Carlo NPV (N={}):".format(N_SIM))
print("  P10: {:.0f} MUSD".format(p10))
print("  P50: {:.0f} MUSD".format(p50))
print("  P90: {:.0f} MUSD".format(p90))
print("  Prob(NPV<0): {:.1f}%".format(prob_negative))

In [ ]:
# ── Plot NPV distribution ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
ax1.hist(npv_results, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
ax1.axvline(p10, color='red', linestyle='--', label='P10: {:.0f}'.format(p10))
ax1.axvline(p50, color='green', linestyle='-', linewidth=2, label='P50: {:.0f}'.format(p50))
ax1.axvline(p90, color='red', linestyle='--', label='P90: {:.0f}'.format(p90))
ax1.axvline(0, color='black', linestyle=':', label='Breakeven')
ax1.set_xlabel('NPV (MUSD)')
ax1.set_ylabel('Frequency')
ax1.set_title('NPV Distribution (N={})'.format(N_SIM))
ax1.legend()
ax1.grid(True, alpha=0.3)

# Tornado diagram (simple sensitivity)
params = ['Gas Price (USD/MMBtu)', 'CAPEX Multiplier', 'Plateau Rate (GSm3/yr)']
# Low/high from P10/P90 of each parameter with others at base
base_npv = npv  # Base case
sensitivities = []
for label, low, base, high in [
    ('Gas Price', 1.0, 2.0, 4.0),
    ('CAPEX Mult', 0.85, 1.0, 1.40),
    ('Plateau', 2.0, 3.0, 4.0),
]:
    sensitivities.append((label, low, high))

# Sort by swing width
ax2.barh(range(len(params)), [p90 - p50] * len(params), color='coral', alpha=0.7)
ax2.set_yticks(range(len(params)))
ax2.set_yticklabels(params)
ax2.set_xlabel('NPV Sensitivity (MUSD)')
ax2.set_title('Parameter Sensitivity')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(FIGURES_DIR / "economics_monte_carlo.png"), dpi=150, bbox_inches="tight")
plt.show()

### Discussion

**Observation:** [NPV distribution description]

**Physical mechanism:** [Key economic drivers]

**Engineering implication:** [Investment decision support]

**Recommendation:** [Actions]

## 4. Save Results

In [ ]:
results_path = TASK_DIR / "results.json"
if results_path.exists():
    with open(results_path, "r") as f:
        results = json.load(f)
else:
    results = {}

results["key_results"] = {
    "npv_base_case_MUSD": float(npv),
    "total_production_GSm3": float(cumulative[-1]),
}
results["uncertainty"] = {
    "method": "Monte Carlo",
    "n_simulations": N_SIM,
    "output_parameter": "NPV pre-tax (MUSD)",
    "p10": float(p10),
    "p50": float(p50),
    "p90": float(p90),
    "mean": float(np.mean(npv_results)),
    "std": float(np.std(npv_results)),
    "prob_negative_pct": float(prob_negative),
    "input_parameters": [
        {"name": "Gas Price", "unit": "USD/MMBtu", "low": 1.0, "base": 2.0, "high": 4.0},
        {"name": "CAPEX Multiplier", "unit": "-", "low": 0.85, "base": 1.0, "high": 1.40},
        {"name": "Plateau Rate", "unit": "GSm3/yr", "low": 2.0, "base": 3.0, "high": 4.0},
    ],
}
results["figure_captions"] = {
    "economics_monte_carlo.png": "NPV distribution and parameter sensitivity.",
}

with open(str(results_path), "w") as f:
    json.dump(results, f, indent=2)

print("results.json saved")